In [58]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import plotly.express as px

In [59]:
data = fetch_california_housing()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name=data.target_names[0])

In [60]:
df = X.copy()
df["HousePrice"] = y
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HousePrice
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


### Median Income is measured in "tens of thousands US dollars" and our target variable is measures in "hundreds of thousands of US dollars"

In [61]:
df["MedInc"] = np.round(df["MedInc"] * 10, 2)
df["HousePrice"] = np.round(df["HousePrice"] * 100, 2)

In [62]:
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HousePrice
0,83.25,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,452.6
1,83.01,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,358.5
2,72.57,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,352.1
3,56.43,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,341.3
4,38.46,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,342.2
...,...,...,...,...,...,...,...,...,...
20635,15.60,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,78.1
20636,25.57,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,77.1
20637,17.00,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,92.3
20638,18.67,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,84.7


### Problem: target house prices are capped at $500.000 (max(500.00, house_price)). Hence this huge imbalance. Will remove them so model doesnt get confused with artificially altered prices

In [63]:
fig = px.histogram(
    x=df["HousePrice"],
    nbins=100,
    labels={
        "x": "House Price ($100k)",
        "y": "Number of Houses"
    },
    title="Distribution of House Prices"
)

fig.show()

In [64]:
df["HousePrice"].value_counts()

HousePrice
500.0    992
137.5    122
162.5    117
112.5    103
187.5     93
        ... 
354.9      1
433.0      1
307.8      1
304.9      1
47.0       1
Name: count, Length: 3841, dtype: int64

In [65]:
indices = (index for index, val in df["HousePrice"].items() if val == 500.000)

df = df.drop(index=indices)

In [66]:
df.to_parquet("california_housing_prices.parquet", index=False)